# CVTD — Stage A experiments on Colab A100 (M0 baseline + M1 SFT)

Runs the real Viettel KPI experiments on **A100** (transformers + PEFT QLoRA, Qwen3-4B).

**Before running** — the real data is gitignored, so stage it to Drive:
1. Locally build the data bundle:
   ```bash
   cd <repo> && python3 scripts/build_real_sft.py --warmup-n 10000 --eval-n 256
   ```
   This (re)creates `drive_upload/data/` style files. The ready-to-upload folder is **`drive_upload/data/`** (18 files, ~53M; see `drive_upload/README.md`).
2. Upload the **`drive_upload/data/` folder** to Google Drive at `MyDrive/cvtd/data/`.
3. Run the cells below (set `REPO_URL` and `DRIVE_DATA`). Cell 2 copies the folder into the repo's `data/`.

(Cell 3 re-runs `build_real_sft.py` for reproducibility; `public_warmup_all.jsonl` is in the folder so the warmup mix reproduces. Skip cell 3 if you trust the prebuilt merged file.)

In [ ]:
# 1) Setup — clone repo + install GPU training stack
REPO_URL = "https://github.com/<you>/telco_function_calling.git"  # <-- set
import os
if not os.path.isdir("telco_function_calling"):
    !git clone $REPO_URL
%cd telco_function_calling
!pip -q install -U transformers datasets peft trl bitsandbytes accelerate openpyxl
!nvidia-smi -L

In [ ]:
# 2) Get data — mount Drive, copy the uploaded data/ folder into the repo
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DATA = '/content/drive/MyDrive/cvtd/data'   # <-- folder you uploaded (drive_upload/data)
!mkdir -p data && cp -r "$DRIVE_DATA"/* data/
!ls -la data/ | head -40
# (zip alternative) !unzip -o /content/drive/MyDrive/cvtd/data_real.zip -d .

In [ ]:
# 3) A1 — (re)build the merged SFT messages file (skip if already in the zip)
!python3 scripts/build_real_sft.py --warmup-n 10000 --eval-n 256
!python3 - <<'PY'
import json
tr=[json.loads(l) for l in open('data/sft_train_real_with_warmup.jsonl')]
dom=[s for s in tr if s.get('source')=='real_tool_xlsx']
assert not any('<from_step_1>' in s['messages'][-1]['content'] for s in tr)
print('train', len(tr), 'real-domain', len(dom))
PY

In [ ]:
# 4) A2 — M0 prompt-only baseline (zero-shot, real splits)
!python3 scripts/run_baseline.py --backend transformers --model Qwen/Qwen3-4B \
    --load-in-4bit --splits real --output reports/m0_real.jsonl

In [ ]:
# 5) A3 — M1 SFT (QLoRA §11.2, single-stage merged warmup+domain)
!python3 scripts/train_sft.py --model Qwen/Qwen3-4B \
    --train-file data/sft_train_real_with_warmup.jsonl --eval-file data/sft_eval_real_messages.jsonl \
    --lora-r 32 --lora-alpha 64 --lora-dropout 0.05 \
    --learning-rate 1e-4 --epochs 1 --max-seq-length 4096 \
    --grad-accum-steps 8 --warmup-ratio 0.03 --weight-decay 0.01 \
    --load-in-4bit --bf16 --output-dir outputs/sft/qwen3-4b-real-m1

In [ ]:
# 6) A4 — M1 eval on real splits
!python3 scripts/run_baseline.py --backend transformers --model Qwen/Qwen3-4B \
    --load-in-4bit --adapter outputs/sft/qwen3-4b-real-m1 --splits real --output reports/m1_real.jsonl

In [ ]:
# 7) Compare M0 vs M1 per split -> reports/real_experiments.md
import json, collections, statistics, pathlib
def agg(path):
    by=collections.defaultdict(list)
    for l in open(path):
        r=json.loads(l); by[r['split']].append(r)
    out={}
    for sp,rs in by.items():
        m=lambda k: round(statistics.mean([x['metrics'].get(k,0.0) for x in rs]),3)
        out[sp]={'n':len(rs),'reward':round(statistics.mean([x['reward'] for x in rs]),3),
                 'func_acc':m('function_selection_accuracy'),'schema':m('schema_validity')}
    return out
m0,m1=agg('reports/m0_real.jsonl'),agg('reports/m1_real.jsonl')
lines=['# Real-tool experiments (Colab A100)\n','| split | n | M0 reward | M1 reward | \u0394 | M0 func | M1 func |','|---|---|---|---|---|---|---|']
for sp in sorted(set(m0)|set(m1)):
    a,b=m0.get(sp,{}),m1.get(sp,{})
    d=round(b.get('reward',0)-a.get('reward',0),3)
    lines.append(f"| {sp} | {b.get('n',a.get('n','-'))} | {a.get('reward','-')} | {b.get('reward','-')} | {d:+} | {a.get('func_acc','-')} | {b.get('func_acc','-')} |")
md='\n'.join(lines)+'\n'
pathlib.Path('reports').mkdir(exist_ok=True); open('reports/real_experiments.md','w').write(md)
print(md)

In [ ]:
# 8) Persist adapter + reports back to Drive
!mkdir -p /content/drive/MyDrive/cvtd/outputs
!cp -r outputs/sft/qwen3-4b-real-m1 /content/drive/MyDrive/cvtd/outputs/
!cp reports/m0_real.jsonl reports/m1_real.jsonl reports/real_experiments.md /content/drive/MyDrive/cvtd/
print('saved to Drive: MyDrive/cvtd/')